# Week 1: data ingestion and cleaning

This notebook streams the official Dev Data Lab judicial case files for 2017?2018, filters at read time to five state codes, builds the right-censored survival table, and writes `data/processed/cases_clean.parquet` without loading the full cohort into memory.

The source state codes for the selected states are Maharashtra (`01`), Karnataka (`03`), Bihar (`08`), Tamil Nadu (`10`), and Uttar Pradesh (`13`). The official `cases_state_key.dta` maps these to PC11 state IDs 27, 29, 10, 33, and 09 respectively. Update `STATE_CODES` if the downloaded files use a different representation.


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import iter_stata_cases, list_available_years
from src.preprocessing import (
    build_survival_table,
    clean_case_types,
    encode_disposal_type,
)

YEARS = [2017, 2018]
STATE_CODES = ["01", "03", "08", "10", "13"]
WINDOW_END = "2018-12-31"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "cases_clean.parquet"
TEMP_PATH = PROCESSED_PATH.with_suffix(".parquet.tmp")
LOAD_COLUMNS = [
    "ddl_case_id", "ddl_filing_judge_id", "ddl_decision_judge_id",
    "state", "state_code", "state_name",
    "district", "district_name", "district_code",
    "court", "court_name", "court_no",
    "filing_date", "decision_date", "date_of_filing", "date_of_decision",
    "year", "filing_year", "filing_month",
    "disp_name", "type_name", "judge_desg", "judge_position",
    "def_name_female", "pet_name_female", "def_adv_female", "pet_adv_female",
    "criminal", "bailable_ipc", "number_sections_ipc",
]

available_years = list_available_years(RAW_DIR)
if not set(YEARS).issubset(available_years):
    raise FileNotFoundError(
        f"Download cases_2017.dta and cases_2018.dta below {RAW_DIR}; "
        f"available years: {available_years}"
    )
print({"raw_dir": str(RAW_DIR), "years": available_years})


## Load the selected cohort

`iter_stata_cases` filters each Stata chunk before the chunk is processed and appended to parquet. This keeps the full national files and the full filtered cohort out of memory.


In [ ]:
print("Streaming the selected state cohort in bounded-memory chunks")
case_chunks = iter_stata_cases(
    YEARS, STATE_CODES, raw_dir=RAW_DIR, columns=LOAD_COLUMNS
)


## Normalize source labels

The public files and their lookup-merged variants use slightly different names. The aliases below preserve the project-wide canonical names consumed by `src.preprocessing`. If a downloaded file uses another name, add it to the relevant tuple rather than changing the downstream functions.

In [ ]:
ALIASES = {
    "filing_date": ("filing_date", "date_filing", "filing_dt", "date_of_filing"),
    "decision_date": ("decision_date", "date_decision", "decision_dt", "date_of_decision"),
    "disposal_type": ("disposal_type", "disp_name", "disposition", "disposition_type"),
    "case_type": ("case_type", "type_name", "case_type_name", "purpose_name"),
}

LOOKUPS = {}

def decode_lookup(frame, code_column, lookup_filename, label_column):
    lookup_path = RAW_DIR / "keys" / lookup_filename
    if code_column not in frame or "year" not in frame or not lookup_path.exists():
        return frame
    if lookup_filename not in LOOKUPS:
        lookup = pd.read_stata(
            lookup_path,
            columns=["year", code_column, label_column],
            convert_categoricals=False,
        )
        lookup["_lookup_year"] = pd.to_numeric(lookup["year"], errors="coerce")
        lookup["_lookup_code"] = pd.to_numeric(lookup[code_column], errors="coerce")
        LOOKUPS[lookup_filename] = lookup.drop(columns=["year", code_column]).drop_duplicates(
            ["_lookup_year", "_lookup_code"]
        )
    result = frame.copy()
    result["_lookup_year"] = pd.to_numeric(result["year"], errors="coerce")
    result["_lookup_code"] = pd.to_numeric(result[code_column], errors="coerce")
    result = result.merge(
        LOOKUPS[lookup_filename],
        on=["_lookup_year", "_lookup_code"],
        how="left",
        validate="many_to_one",
    )
    result[code_column] = result[label_column].where(
        result[label_column].notna(), result[code_column]
    )
    return result.drop(columns=["_lookup_year", "_lookup_code", label_column])


In [ ]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
if TEMP_PATH.exists():
    TEMP_PATH.unlink()

writer = None
schema = None
total_rows = 0
total_events = 0
dropped_rows = 0
missing_counts = None
category_counts = {"disposal_type": {}, "case_type": {}}

try:
    for raw in case_chunks:
        source_rows = len(raw)
        for code_column, filename, label_column in (
            ("disp_name", "disp_name_key.dta", "disp_name_s"),
            ("type_name", "type_name_key.dta", "type_name_s"),
        ):
            raw = decode_lookup(raw, code_column, filename, label_column)

        rename_map = {}
        missing = []
        for canonical, candidates in ALIASES.items():
            source = next((column for column in candidates if column in raw.columns), None)
            if source is None:
                missing.append((canonical, candidates))
            elif source != canonical:
                rename_map[source] = canonical
        if missing:
            raise KeyError(f"Could not find required source columns: {missing}")

        cleaned = raw.rename(columns=rename_map)
        cleaned = encode_disposal_type(cleaned)
        cleaned = clean_case_types(cleaned)
        cleaned = build_survival_table(cleaned, window_end=WINDOW_END)
        dropped_rows += source_rows - len(cleaned)
        if cleaned.empty:
            continue
        for column in cleaned.select_dtypes(include=["object"]).columns:
            cleaned[column] = cleaned[column].astype("string")
        table = pa.Table.from_pandas(cleaned, preserve_index=False)
        if writer is None:
            schema = table.schema
            writer = pq.ParquetWriter(TEMP_PATH, schema, compression="snappy")
        elif table.schema != schema:
            table = table.cast(schema, safe=False)
        writer.write_table(table)

        total_rows += len(cleaned)
        total_events += int(cleaned["event"].sum())
        if missing_counts is None:
            missing_counts = pd.Series(0, index=cleaned.columns, dtype="int64")
        missing_counts = missing_counts.add(cleaned.isna().sum(), fill_value=0)
        for column in category_counts:
            for value, count in cleaned[column].value_counts(dropna=False).items():
                category_counts[column][value] = category_counts[column].get(value, 0) + int(count)
finally:
    if writer is not None:
        writer.close()

if writer is None:
    raise RuntimeError("Filtered ingestion produced no rows")
os.replace(TEMP_PATH, PROCESSED_PATH)
summary = pd.Series({
    "rows": total_rows,
    "events": total_events,
    "censored": total_rows - total_events,
    "dropped_invalid_rows": dropped_rows,
    "censoring_rate": (total_rows - total_events) / total_rows,
})
print(f"Wrote {total_rows:,} rows to {PROCESSED_PATH}")


## Sanity checks

Pending cases have `event == 0` and are censored at `WINDOW_END`; disposed cases have `event == 1`.

In [ ]:
display(summary.to_frame("value"))
display(pd.DataFrame(category_counts).fillna(0).astype(int))
display((missing_counts / total_rows).sort_values(ascending=False).head(15).to_frame("missing_rate"))


In [ ]:
assert PROCESSED_PATH.exists()
print(f"Validated output path: {PROCESSED_PATH}")
